# PyTorch product-price embedding demo (learning notebook)

**What this is.** A personal PyTorch learning exercise, built while learning
embeddings, using Curalina's real supplier catalogue data (Celadon, Lazzoni,
Luxus) as a realistic, non-trivial dataset to practice on.

**What this is not.** This is *not* one of the project's formal evaluation
notebooks (`R01`-`R03`, `V01`-`V03`, `G01`-`G03`, `D01`) and does not follow
the seven-section notebook standard (`agentic_flow/16_notebook_standard.md`).
It carries no accuracy, feasibility, or acceptance claim, and it does not
feed into or override `ADR-0006` (the ranking-evaluation leakage firewall) or
`ADR-0013` (the R03 no-go on real-furniture-data *bundle composition*). It
freely uses `room_type`/`design_style`/`atmosphere` as model inputs, which
the real recommendation service's `Product` domain object deliberately
excludes for evaluation-integrity reasons that don't apply here — this
notebook is training a toy price-prediction model, not ranking products.

**The task.** Predict a product's retail price from its catalogue
attributes (supplier, room type, design style, atmosphere, dimensions).
This is a genuine supervised-learning task on real labels (the actual
prices), not a fabricated one — it exists to practice `nn.Embedding` and a
standard PyTorch training loop, in the same style as your other notebooks.

## Running locally vs. in Google Colab

**Local (default):** `DATA_DIR` below points straight at the three supplier
workbooks on disk. Nothing else to do.

**Google Colab:** upload the `Supplier CSV Files` folder to your Google
Drive, then run this in a cell *before* the data-loading cell, and change
`DATA_DIR` to match:

```python
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/Supplier CSV Files')
```

Colab also needs one extra install cell (uncomment the line in the next
cell) — locally, `pip install torch pandas openpyxl matplotlib` once,
outside the notebook, is enough.


In [ ]:
# Uncomment in Colab (locally, install these once from your terminal instead):
# !pip install torch pandas openpyxl matplotlib

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# --- Path configuration --------------------------------------------------
# Local default. In Colab, mount Drive first (see markdown above) and
# reassign this to the Drive path instead.
DATA_DIR = Path(os.environ.get(
    "CURALINA_SUPPLIER_DATA_DIR",
    "/Users/rjsalmon/Documents/Humber/misc.curalina/Supplier CSV Files",
))

WORKBOOKS = {
    "Celadon": DATA_DIR / "Celadon CSV Programmer Handoff.xlsx",
    "Lazzoni": DATA_DIR / "Lazzoni CSV Programmer Handoff.xlsx",
    "Luxus": DATA_DIR / "Luxus Programmer Handoff.xlsx",
}
for supplier, path in WORKBOOKS.items():
    print(supplier, "->", path, "exists:", path.exists())


## 1. Load and clean the three supplier workbooks

These are named "CSV" files but are actually `.xlsx`. Worse for us: **the
three suppliers don't use the same column names or the same column order**
(confirmed by inspecting the headers directly — e.g. Celadon calls its key
column `SKU`, Luxus calls the same idea `SUPPLIER SKU`, and Luxus leaves it
blank for most rows entirely). So the first real step is a small
column-alias map that renames each workbook's headers onto one shared,
canonical schema before we concatenate them.


In [ ]:
# Canonical column name -> the possible header spellings we've actually seen.
COLUMN_ALIASES = {
    "sku": ["SKU", "Supplier SKU", "SUPPLIER SKU"],
    "product_name": ["Product Name"],
    "supplier": ["Supplier"],
    "trade_price": ["Trade Price"],
    "retail_price": ["Retail Price"],
    "room_type": ["Room Type"],
    "design_style": ["Design Style"],
    "atmosphere": ["Atmosphere"],
    "width_in": ["Width (in)"],
    "depth_in": ["Depth (in)"],
    "height_in": ["Height (in)"],
    "overview": ["Product Overview"],
}


def load_and_rename(path: Path, supplier_label: str) -> pd.DataFrame:
    df = pd.read_excel(path, engine="openpyxl")
    rename_map = {}
    for canonical, aliases in COLUMN_ALIASES.items():
        for alias in aliases:
            if alias in df.columns:
                rename_map[alias] = canonical
                break
    df = df.rename(columns=rename_map)
    # Keep only the columns we know how to rename; a workbook missing one
    # entirely (e.g. Celadon has no depth_in — artwork is flat) just gets
    # that column filled with NaN rather than crashing.
    for canonical in COLUMN_ALIASES:
        if canonical not in df.columns:
            df[canonical] = np.nan
    df = df[list(COLUMN_ALIASES.keys())].copy()
    df["supplier"] = supplier_label  # trust the filename, not the sheet
    return df


raw_frames = [load_and_rename(path, supplier) for supplier, path in WORKBOOKS.items()]
catalogue = pd.concat(raw_frames, ignore_index=True)
print(f"Loaded {len(catalogue)} rows across {len(WORKBOOKS)} suppliers.")
catalogue.head()


### Cleaning steps, explained

A few real, specific problems in this data that a generic `dropna()` would
either miss or over-correct:

1. **Missing SKUs (Luxus).** Many Luxus rows have a blank SKU — we still
   need a stable per-row identifier, so we fall back to the row's index
   when the SKU is missing, rather than dropping the row.
2. **`retail_price == 0` is missing, not free.** A handful of rows (mostly
   Luxus) have `retail_price = 0` with no `trade_price` either — that's a
   data-entry placeholder, not a real price of zero. We treat `0` as
   missing for both price columns.
3. **`room_type`, `design_style`, and `atmosphere` are all multi-valued**
   (e.g. `"Mid-Century Scandinavian; Contemporary Luxe"` for a piece that
   fits two styles). For this simple demo we only take the *first* listed
   tag for each — good enough to practice embeddings on, not something
   you'd ship.
4. **Rows with no price at all get dropped.** Price is our training target,
   so a row we can't price is useless to us here (this is a modelling
   choice for a demo, not a data-quality judgement about those products).


In [ ]:
def first_tag(value):
    if not isinstance(value, str):
        return "unknown"
    return value.split(";")[0].strip() or "unknown"


clean = catalogue.copy()
clean["sku"] = clean["sku"].astype("object")
clean.loc[clean["sku"].isna(), "sku"] = [
    f"row_{i}" for i in clean.index[clean["sku"].isna()]
]

for price_col in ["trade_price", "retail_price"]:
    clean[price_col] = pd.to_numeric(clean[price_col], errors="coerce")
    clean.loc[clean[price_col] == 0, price_col] = np.nan

for tag_col in ["room_type", "design_style", "atmosphere"]:
    clean[tag_col] = clean[tag_col].apply(first_tag)

for dim_col in ["width_in", "depth_in", "height_in"]:
    clean[dim_col] = pd.to_numeric(clean[dim_col], errors="coerce")
    clean[dim_col] = clean[dim_col].fillna(clean[dim_col].median())

clean = clean.dropna(subset=["retail_price"]).reset_index(drop=True)
print(f"{len(clean)} rows left after cleaning (target = retail_price).")
clean[["sku", "supplier", "room_type", "design_style", "atmosphere", "retail_price"]].head()


## 2. Embeddings, explained

Our categorical columns (`supplier`, `room_type`, `design_style`,
`atmosphere`) are strings — a neural net needs numbers. The naive fix is
one-hot encoding, but that treats every category as equally different from
every other one, and doesn't scale once a column has hundreds of values.

`nn.Embedding` instead learns a small dense vector *per category value*
during training — e.g. every `design_style` becomes an 8-number vector, and
similar styles can end up with similar vectors, the same way word
embeddings work in NLP. Mechanically, an embedding layer is just a lookup
table: `nn.Embedding(num_categories, embedding_dim)` stores one trainable
vector per category, and indexing it with a category's integer id returns
that vector.

So the recipe is:

1. Build a vocabulary (a `dict` mapping each category string to an integer
   id) for each categorical column.
2. Convert every row's category strings to their integer ids.
3. In the model, look each id up in its own `nn.Embedding` table, then
   concatenate all the embedding vectors together with the numeric
   (continuous) features before the first `nn.Linear` layer.


In [ ]:
CATEGORICAL_COLUMNS = ["supplier", "room_type", "design_style", "atmosphere"]
CONTINUOUS_COLUMNS = ["width_in", "depth_in", "height_in"]

vocabularies = {}
for col in CATEGORICAL_COLUMNS:
    unique_values = sorted(clean[col].unique())
    vocabularies[col] = {value: idx for idx, value in enumerate(unique_values)}
    print(f"{col}: {len(unique_values)} unique values")

for col in CATEGORICAL_COLUMNS:
    clean[f"{col}_id"] = clean[col].map(vocabularies[col])


## 3. Build tensors, train/test split, `DataLoader`

Same shape as your other notebooks: a `TensorDataset` + `DataLoader` per
split. The only difference is we now have *two* input tensors per example
(one `LongTensor` of category ids, one `FloatTensor` of continuous
features) instead of one — `TensorDataset` happily takes both, and
`DataLoader` yields a three-tuple per batch (`categorical, continuous,
target`) instead of the usual pair.


In [ ]:
torch.manual_seed(42)

X_categorical = torch.tensor(clean[[f"{c}_id" for c in CATEGORICAL_COLUMNS]].values, dtype=torch.long)
X_continuous = torch.tensor(clean[CONTINUOUS_COLUMNS].values, dtype=torch.float32)
y = torch.tensor(clean[["retail_price"]].values, dtype=torch.float32)

# Standardize the continuous features and the target -- an MLP trains much
# more reliably on roughly unit-scale numbers than on raw dollar amounts.
cont_mean, cont_std = X_continuous.mean(dim=0), X_continuous.std(dim=0).clamp(min=1e-6)
X_continuous = (X_continuous - cont_mean) / cont_std
y_mean, y_std = y.mean(), y.std()
y = (y - y_mean) / y_std

n = len(clean)
n_train = int(n * 0.8)
perm = torch.randperm(n)
train_idx, test_idx = perm[:n_train], perm[n_train:]

train_dataset = TensorDataset(X_categorical[train_idx], X_continuous[train_idx], y[train_idx])
test_dataset = TensorDataset(X_categorical[test_idx], X_continuous[test_idx], y[test_idx])
train_dl = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_dl = DataLoader(test_dataset, batch_size=8, shuffle=True)

print(f"train: {len(train_dataset)} rows, test: {len(test_dataset)} rows")


## 4. The model

Same two-layer MLP shape as `SimpleLinearRegression` (`fc1` -> `relu` ->
`fc2`), just with an embedding table per categorical column feeding into
`fc1` alongside the continuous features.


In [ ]:
class ProductPriceModel(nn.Module):
    def __init__(self, vocab_sizes: dict[str, int], n_continuous: int, embedding_dim: int = 8):
        super().__init__()
        self.embeddings = nn.ModuleDict({
            col: nn.Embedding(vocab_size, embedding_dim)
            for col, vocab_size in vocab_sizes.items()
        })
        total_in = embedding_dim * len(vocab_sizes) + n_continuous
        self.fc1 = nn.Linear(total_in, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x_categorical, x_continuous):
        embedded = [
            self.embeddings[col](x_categorical[:, i])
            for i, col in enumerate(CATEGORICAL_COLUMNS)
        ]
        x = torch.cat([*embedded, x_continuous], dim=1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

    def predict(self, x_categorical, x_continuous):
        with torch.no_grad():
            result = self.forward(x_categorical, x_continuous)
        return result


device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

vocab_sizes = {col: len(vocabularies[col]) for col in CATEGORICAL_COLUMNS}
model = ProductPriceModel(vocab_sizes, n_continuous=len(CONTINUOUS_COLUMNS)).to(device)
model


## 5. Training loop

Same shape you've been using: MSE loss, Adam, one `losses` list per epoch.
We also track the test-set loss each epoch (in `model.eval()` +
`torch.no_grad()`, so it never influences the gradients) so we can plot
train vs. test loss below and see over/under-fitting directly, and we track
the optimizer's learning rate too (constant here, but this is the hook
you'd use if you added a scheduler later).


In [ ]:
epochs = 100
train_losses = []
test_losses = []
learning_rates = []

criterion = nn.MSELoss()
torch.manual_seed(42)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for batch_cat, batch_cont, batch_y in train_dl:
        batch_cat = batch_cat.to(device)
        batch_cont = batch_cont.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        y_pred = model(batch_cat, batch_cont)
        loss = criterion(y_pred, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_train_loss = epoch_loss / len(train_dl)
    train_losses.append(avg_train_loss)

    model.eval()
    with torch.no_grad():
        test_loss = 0.0
        for batch_cat, batch_cont, batch_y in test_dl:
            batch_cat, batch_cont, batch_y = batch_cat.to(device), batch_cont.to(device), batch_y.to(device)
            y_pred = model(batch_cat, batch_cont)
            test_loss += criterion(y_pred, batch_y).item()
        avg_test_loss = test_loss / len(test_dl)
    test_losses.append(avg_test_loss)

    learning_rates.append(optimizer.param_groups[0]["lr"])

    if (epoch + 1) % 10 == 0:
        print(f"Epoch: {epoch+1}, Train Loss: {avg_train_loss:.4f}, Test Loss: {avg_test_loss:.4f}")


## 6. Plots

Three plots, academic-report style (labelled axes, title, legend, grid):
train vs. test loss over epochs, the learning rate (flat here since we're
not using a scheduler, but this is where you'd see it change if you added
one), and a predicted-vs-actual scatter on the held-out test set as a
sanity check on the errors themselves, not just the aggregate loss number.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(train_losses, label="train loss")
axes[0].plot(test_losses, label="test loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE loss (standardized price)")
axes[0].set_title("Training and test loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(learning_rates)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Learning rate")
axes[1].set_title("Learning rate schedule")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()


In [ ]:
model.eval()
with torch.no_grad():
    all_cat = X_categorical[test_idx].to(device)
    all_cont = X_continuous[test_idx].to(device)
    preds_standardized = model.predict(all_cat, all_cont).cpu()

# Undo the standardization from section 3 so the plot is in real dollars.
preds_dollars = (preds_standardized * y_std + y_mean).numpy().flatten()
actual_dollars = (y[test_idx] * y_std + y_mean).numpy().flatten()

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(actual_dollars, preds_dollars, alpha=0.6)
lims = [min(actual_dollars.min(), preds_dollars.min()), max(actual_dollars.max(), preds_dollars.max())]
ax.plot(lims, lims, "r--", label="perfect prediction")
ax.set_xlabel("Actual retail price ($)")
ax.set_ylabel("Predicted retail price ($)")
ax.set_title("Predicted vs. actual price (test set)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

mae = float(np.mean(np.abs(preds_dollars - actual_dollars)))
print(f"Test MAE: ${mae:,.2f}")


## What this does and doesn't show you

This demonstrates the PyTorch mechanics (embeddings, `TensorDataset` +
`DataLoader`, a standard training loop, loss/LR/error plotting) on real
Curalina catalogue data. It is **not** the recommendation service's actual
ranking model, does not use the service's real `Product`/`Money` domain
objects, and its price-prediction task has nothing to do with what
`recommendation`'s `/v1/recommendations` endpoint needs to do (rank
candidate products against a design profile, not predict their price).

If a real embedding-based ranking or recommendation model is wanted for the
actual service, that decision — architecture, evaluation methodology,
accept/reject — is `ai-ml-lead`'s call per this project's roster
(`AGENTS.md`), following the same evidence-notebook standard as `R01`-`R03`.
This notebook is a stepping stone toward being able to read and reason
about that work, not a substitute for it.
